In [1]:
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load cleaned data
df = pd.read_csv('filtered_complaints.csv')
df.head()

,Unnamed: 0,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,cleaned_narrative
0,76,2025-03-06,Credit reporting or other personal consumer re...,Credit reporting,Problem with fraud alerts or security freezes,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,75211,NaN,Consent provided,Web,2025-03-06,Closed with explanation,Yes,NaN,12351447,xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx apt xx...
1,359,2025-02-26,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information is missing that should be on the r...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,331XX,NaN,Consent provided,Web,2025-02-26,Closed with explanation,Yes,NaN,12203565,subject dispute of unauthorized hard inquiries...
2,11499,2025-06-15,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an exis...,Changes in loan terms during the application p...,Company has responded to the consumer and the ...,"Lennar Financial Services, LLC",IL,60538,NaN,Consent provided,Web,2025-06-15,Closed with explanation,Yes,NaN,14089944,i signed a purchase agreement with lennar corp...
3,11973,2025-06-14,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,Company has responded to the consumer and the ...,"Fidelity National Information Services, Inc. (...",FL,32303,NaN,Consent provided,Web,2025-06-14,Closed with explanation,Yes,NaN,14080390,after checking my report i found numerous acco...
4,12237,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,Servicemember,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,a xxxx xxxx card was opened under my name by a...


In [2]:
# Extract the cleaned narratives
texts = df['cleaned_narrative'].dropna().tolist()

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

# Split each narrative into chunks
all_chunks = []
for text in texts:
    chunks = text_splitter.split_text(text)
    all_chunks.extend(chunks)

# Preview result
print(f"Total chunks created: {len(all_chunks)}")
print(all_chunks[:5])  


Total chunks created: 7612874
['xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx apt xxxx xxxx tx xxxx xxxx xxxxxxxx transunion consumer solutions xxxx xxxx xxxx xxxx pa xxxx xxxx xxxx re security freeze request dear sirmadam i xxxx xxxx xxxx xxxx xxxx xxxx social security xxxx xxxx a resident of xxxx xxxx xxxx xxxxxxxx xxxx xxxx xxxx tx xxxx submit this affidavit to request a security freeze on my credit report pursuant to my rights under applicable federal and state laws including but not limited to the fair credit reporting act and', 'not limited to the fair credit reporting act and any other relevant regulations i hereby formally request that a security freeze be placed on my credit file immediately affidavit applicant identification full name xxxx xxxx xxxx xxxx social security number xxxx date of birth xxxxxxxx current address xxxx xxxx xxxx xxxx apt xxxx xxxx xxxx tx xxxx xxxx xxxx request o i request that a security freeze be placed on my credit report to prevent unauthorized thirdparty

In [3]:
texts = df['cleaned_narrative'].dropna().tolist()

# Define configurations to test
configs = [
    {'chunk_size': 300, 'chunk_overlap': 50},
    {'chunk_size': 500, 'chunk_overlap': 50},
    {'chunk_size': 700, 'chunk_overlap': 100},
]

# Test each configuration
for config in configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config['chunk_size'],
        chunk_overlap=config['chunk_overlap'],
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    total_chunks = 0
    for text in texts:
        chunks = splitter.split_text(text)
        total_chunks += len(chunks)
    
    print(f"chunk_size={config['chunk_size']}, chunk_overlap={config['chunk_overlap']} → Total Chunks: {total_chunks}")


chunk_size=300, chunk_overlap=50 → Total Chunks: 12413451
chunk_size=500, chunk_overlap=50 → Total Chunks: 7612874
chunk_size=700, chunk_overlap=100 → Total Chunks: 5958547


Smaller chunk_size → more chunks, better granularity, but more noise.

Larger chunk_overlap → better context continuity across chunks, but increased redundancy and processing cost.

A balanced setup often used: chunk_size=500, chunk_overlap=50.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')


# Generate embeddings
embeddings = model.encode(all_chunks, show_progress_bar=True)

# Convert embeddings to NumPy array
embedding_matrix = np.array(embeddings).astype('float32')  # FAISS requires float32

# Get the dimensionality of the vectors
dimension = embedding_matrix.shape[1]  # Should be 384 for all-MiniLM-L6-v2

# Create a FAISS index (L2 = Euclidean distance)
index = faiss.IndexFlatL2(dimension)

# Add embeddings to the index
index.add(embedding_matrix)

# Save the index
faiss.write_index(index, 'complaints_index.faiss')

print(f"FAISS index created with {index.ntotal} vectors.")


Batches:   0%|          | 0/237903 [00:00<?, ?it/s]